# RenalGuard — Stage 4: Data Preprocessing and Feature Engineering
## Building a Robust, Leakage-Free Preprocessing Pipeline for Chronic Kidney Disease Prediction

Welcome to Stage 4 of **RenalGuard**!

### What is Preprocessing & Feature Engineering?
In Stage 3 (Exploratory Data Analysis), we inspected the raw UCI Chronic Kidney Disease dataset and discovered critical findings:
1. Significant missing data (up to 38% in urinalysis microscopy).
2. Data entry formatting issues (trailing tabs like `'ckd\t'` and leading tabs like `'\tno'`).
3. Mixed data types: 11 continuous biomarkers and 13 categorical clinical variables.
4. Clinically coded discrete variables (`sg`, `al`, `su`) that appear numeric in CSV format but represent qualitative urinalysis dipstick intervals.

Raw medical data cannot be passed directly into mathematical learning algorithms. Machine learning models require **complete**, **numeric**, and **appropriately scaled** feature matrices.

### Crucial Principle for ML Beginners: Preventing Data Leakage
**Data leakage** occurs when information from the test dataset (evaluation data) accidentally influences the training process. 
- *Common Beginner Mistake*: Imputing missing values using the mean/median of the *entire* dataset, or scaling features across all rows before splitting.
- *The Golden Rule*: The test set must simulate unseen future patients in a clinic. **All imputers, encoders, and scalers must be fitted strictly on training data**, then applied to transform test data using those exact learned parameters.


---
## 1. Data Cleaning

**Beginner Explanation**:
- We load the dataset into a working DataFrame without modifying the original file on disk.
- We address missing value representations: in historical datasets, missing entries are sometimes coded as `'?'` or whitespace. We standardize these to `np.nan`.
- We remove accidental leading/trailing whitespace and tabs from column names and text entries.
- We check for duplicate records and investigate their integrity.


In [3]:
import os
import pandas as pd
import numpy as np

# Load original dataset without modifying the source file
data_path = os.path.join('..', 'data', 'chronic_kidney_disease.csv')
df = pd.read_csv(data_path)

print("Original dataset loaded successfully!")
print(f"Raw shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Check duplicates before cleaning
dup_count = df.duplicated().sum()
print(f"Checking duplicates in raw dataset: {dup_count} duplicate rows found.")

# 1. Clean column names (strip whitespace)
df.columns = df.columns.str.strip()

# 2. Standardize missing indicators ('?', whitespace) to NaN
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True).replace(r'^\s*$', np.nan, regex=True)

# 3. Strip accidental whitespace and tabs from categorical string values
str_cols = df.select_dtypes(include=['object', 'string', 'str']).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '?': np.nan})

print("Whitespace stripped from column names and string columns.")
print("Placeholder values ('?', empty strings) standardized to np.nan.")


Original dataset loaded successfully!
Raw shape: 400 rows, 25 columns
Checking duplicates in raw dataset: 0 duplicate rows found.
Whitespace stripped from column names and string columns.
Placeholder values ('?', empty strings) standardized to np.nan.


**Why this step is necessary**:
- Even a single invisible whitespace character (like `'\tno'` in diabetes or `'ckd\t'` in class) causes Python to treat it as an entirely separate category from `'no'` or `'ckd'`.
- Standardizing placeholders ensures Pandas and Scikit-Learn accurately recognize all missing observations as `np.nan`.


---
## 2. Feature Identification & Target Normalization

**Beginner Explanation**:
1. **Target Normalization**: In binary classification, the disease state should be mapped to `1` (positive, has CKD) and the healthy state to `0` (negative, healthy). This ensures standard loss functions (like log-loss) and metrics (like ROC-AUC and Recall) evaluate correctly.
2. **Clinically Coded Categoricals**:
   - In the UCI dataset, three features appear as numbers: Specific Gravity (`sg`), Albumin (`al`), and Sugar (`su`).
   - In clinical pathology, these are **urinalysis dipstick categories**:
     - `sg`: Discrete density bands (1.005, 1.010, 1.015, 1.020, 1.025).
     - `al`: Proteinuria grading scale (0 = nil, 1 = trace/+, 2 = ++, 3 = +++, 4 = ++++, 5 = massive).
     - `su`: Glycosuria grading scale (0 = normal, 1 to 5 = severity).
   - If left as raw continuous numbers, an algorithm assumes the distance between grade 0 and 1 is mathematically identical to grade 4 and 5. Converting them to categorical variables allows One-Hot Encoding without imposing an arbitrary linear relationship.


In [6]:
# Normalize target variable: 'ckd' and 'ckd\t' -> 1, 'notckd' -> 0
df['class'] = df['class'].replace({'ckd': 1, 'notckd': 0}).astype(int)

# Identify clinically coded categorical features and ensure they are formatted as categories
clinically_coded = ['sg', 'al', 'su']
for col in clinically_coded:
    df[col] = df[col].apply(lambda x: str(x) if pd.notna(x) and str(x) != 'nan' else np.nan)

# Define feature lists
numerical_features = ['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']
categorical_features = ['sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']

print("Target normalization complete:")
print(f"Raw target counts: {target_before_cleaning}")
print(f"Normalized target counts: {target_after_cleaning}")
print(f"1 (CKD): {target_after_cleaning.get(1)} patients (62.5%)")
print(f"0 (notckd): {target_after_cleaning.get(0)} patients (37.5%)")
print(f"\nNumerical features identified ({len(numerical_features)}):\n{numerical_features}")
print(f"\nCategorical features identified ({len(categorical_features)}, including 3 clinically coded urinalysis variables):\n{categorical_features}")


Target normalization complete:
Raw target counts: {'ckd': 248, 'notckd': 150, 'ckd\t': 2}
Normalized target counts: {1: 250, 0: 150}
1 (CKD): 250 patients (62.5%)
0 (notckd): 150 patients (37.5%)

Numerical features identified (11):
['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']

Categorical features identified (13, including 3 clinically coded urinalysis variables):
['sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


---
## 3. Train-Test Split (Avoiding Data Leakage)

**Beginner Explanation**:
- We split the dataset into:
  - **Training Set (80%)**: 320 patients used to learn patterns and fit preprocessing parameters.
  - **Testing Set (20%)**: 80 patients held out to evaluate how well our future models generalize to unseen clinical patients.
- **Stratified Splitting**: `stratify=y` ensures both the train and test sets have the exact same disease prevalence (62.5% CKD, 37.5% healthy). Without stratification, random splits could end up with an unrepresentative test set.
- **`random_state=42`**: Fixes the random seed so our split is 100% reproducible every time.


In [8]:
from sklearn.model_selection import train_test_split

# Separate predictors (X) and target outcome (y)
X = df.drop(columns=['class'])
y = df['class']

# Perform 80:20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Total patients: {len(df)}")
print(f"X_train shape: {X_train.shape} (80% of data)")
print(f"X_test shape:  {X_test.shape} (20% of data)")
print(f"\nTraining target distribution: {dict(pd.Series(y_train).value_counts())} (CKD: {y_train.mean()*100:.1f}%)")
print(f"Testing target distribution:  {dict(pd.Series(y_test).value_counts())} (CKD: {y_test.mean()*100:.1f}%)")


Total patients: 400
X_train shape: (320, 24) (80% of data)
X_test shape:  (80, 24) (20% of data)

Training target distribution: {1: np.int64(200), 0: np.int64(120)} (CKD: 62.5%)
Testing target distribution:  {1: np.int64(50), 0: np.int64(30)} (CKD: 62.5%)


**Why splitting before transformation is mandatory**:
If you calculate the mean or median of a feature using all 400 rows, the resulting number incorporates information from the 80 test patients. That is **data leakage**. By splitting first, our imputers and scalers only ever learn from `X_train`.


---
## 4. Missing Value Imputation Strategy

**Beginner Explanation**:
- **Numerical Features $\rightarrow$ Median Imputation (`SimpleImputer(strategy='median')`)**:
  - *Why not the mean?* In Stage 3, we observed that serum creatinine (`sc`) and blood urea (`bu`) have extreme right-skewed outliers (e.g. creatinine values of 76 mg/dL compared to normal 1.0 mg/dL). The arithmetic mean is heavily pulled by outliers, whereas the **median** remains robust and represents typical values.
- **Categorical Features $\rightarrow$ Most Frequent Imputation (`SimpleImputer(strategy='most_frequent')`)**:
  - Replaces missing urinalysis findings and symptoms with the mode (most common value observed in the training cohort).


In [11]:
from sklearn.impute import SimpleImputer

# Numerical imputer: uses median (robust to extreme clinical outliers)
num_imputer = SimpleImputer(strategy='median')

# Categorical imputer: uses most frequent (mode)
cat_imputer = SimpleImputer(strategy='most_frequent')

print("Imputers configured successfully:")
print(" - Numerical: SimpleImputer(strategy='median')")
print(" - Categorical: SimpleImputer(strategy='most_frequent')")


Imputers configured successfully:
 - Numerical: SimpleImputer(strategy='median')
 - Categorical: SimpleImputer(strategy='most_frequent')


---
## 5. Encoding and Feature Scaling

**Beginner Explanation**:
1. **One-Hot Encoding (`OneHotEncoder(handle_unknown='ignore')`)**:
   - Machine learning algorithms cannot perform algebra on text strings like `'normal'` or `'abnormal'`.
   - One-hot encoding creates binary (0 or 1) indicator columns for each category.
   - `handle_unknown='ignore'`: If a future patient presents with an unseen category, the encoder will safely assign all zeros rather than crashing the pipeline.
2. **Feature Scaling (`StandardScaler()`)**:
   - Standardizes numerical features by centering to mean 0 and scaling to unit variance ($z = \frac{x - \mu}{\sigma}$).
   - *Why is this essential?* White blood cell count ranges around $8,000$, while serum creatinine ranges around $1.2$. Without scaling, distance-based and gradient-based algorithms (like Logistic Regression, SVM, KNN, and Neural Networks) would treat white blood cells as $7,000$ times more important simply because of the units!


In [13]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Feature Scaler for numerical columns
scaler = StandardScaler()

# OneHotEncoder for categorical columns (outputting dense array, ignoring unseen categories)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

print("Encoders and Scalers configured successfully:")
print(" - Numerical Scaler: StandardScaler()")
print(" - Categorical Encoder: OneHotEncoder(handle_unknown='ignore', sparse_output=False)")


Encoders and Scalers configured successfully:
 - Numerical Scaler: StandardScaler()
 - Categorical Encoder: OneHotEncoder(handle_unknown='ignore', sparse_output=False)


---
## 6. Building the Reusable Pipeline with `ColumnTransformer`

**Beginner Explanation**:
- **`Pipeline`**: Chains multiple data transformation steps sequentially (e.g., first impute missing values, then scale).
- **`ColumnTransformer`**: Directs different subsets of columns to different pipelines:
  - Numerical columns $\rightarrow$ `num_pipeline` (Median Imputation $\rightarrow$ StandardScaler)
  - Categorical columns $\rightarrow$ `cat_pipeline` (Most Frequent Imputation $\rightarrow$ OneHotEncoder)
- **Why this design is industry standard**:
  - Completely eliminates manual data prep mistakes.
  - Guarantees 0 data leakage.
  - Can be saved (`joblib`) and paired with any classifier (Logistic Regression, Random Forest, XGBoost) in Stage 5.


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Pipeline for numerical features
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline for categorical features
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine both into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_features),
        ('cat', cat_pipeline, categorical_features)
    ],
    remainder='drop',
    verbose_feature_names_out=True
)

print("Reusable Preprocessor Pipeline assembled successfully:")
print(preprocessor)


Reusable Preprocessor Pipeline assembled successfully:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot',
                                  'hemo', 'pcv', 'wbcc', 'rbcc']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba',
                     

---
## 7. Pipeline Execution and Validation

**Beginner Explanation**:
- We call `preprocessor.fit_transform(X_train)`:
  1. `fit`: Computes the medians, means, standard deviations, and category levels from `X_train`.
  2. `transform`: Applies the imputation, scaling, and one-hot encoding to `X_train`.
- We call `preprocessor.transform(X_test)`:
  - Uses the **already fitted parameters from training** to transform the test set without re-fitting.
- We perform strict quality checks:
  1. Verify zero missing values remain.
  2. Check matrix dimensions.
  3. Inspect the transformed feature names.


In [17]:
# Fit on training data ONLY, then transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Retrieve transformed feature names
feature_names = preprocessor.get_feature_names_out().tolist()

# Verification checks
print(f"Training feature matrix shape: {X_train_processed.shape}")
print(f"Testing feature matrix shape:  {X_test_processed.shape}")
print(f"\nMissing values in X_train_processed: {np.isnan(X_train_processed).sum()}")
print(f"Missing values in X_test_processed:  {np.isnan(X_test_processed).sum()}")
print(f"\nTotal transformed features: {len(feature_names)}")
print(f"Numerical features transformed: {len(numerical_features)}")
print(f"One-hot encoded categorical columns: {len(feature_names) - len(numerical_features)}")
print(f"\nFirst 10 transformed feature names:\n{feature_names[:10]}")
print(f"\nCategorical transformed feature names preview:\n{feature_names[11:21]}")


Training feature matrix shape: (320, 48)
Testing feature matrix shape:  (80, 48)

Missing values in X_train_processed: 0
Missing values in X_test_processed:  0

Total transformed features: 48
Numerical features transformed: 11
One-hot encoded categorical columns: 37

First 10 transformed feature names:
['num__age', 'num__bp', 'num__bgr', 'num__bu', 'num__sc', 'num__sod', 'num__pot', 'num__hemo', 'num__pcv', 'num__wbcc']

Categorical transformed feature names preview:
['cat__sg_1.005', 'cat__sg_1.01', 'cat__sg_1.015', 'cat__sg_1.02', 'cat__sg_1.025', 'cat__al_0.0', 'cat__al_1.0', 'cat__al_2.0', 'cat__al_3.0', 'cat__al_4.0']


---
## 8. Summary of the Processed Dataset & Readiness Confirmation

### Final Processed Dataset Condition:
1. **Training Set**: $320$ patients, $48$ engineered and scaled numerical features.
2. **Testing Set**: $80$ patients, $48$ engineered and scaled numerical features.
3. **Target Distribution**: Exactly preserved at **62.5% CKD** ($200$ train / $50$ test) and **37.5% Healthy** ($120$ train / $30$ test) via stratification.
4. **Data Integrity**: **0 missing values**, **0 data leakage**, and full compatibility with Scikit-Learn estimators.
5. **Reusable Module**: The complete pipeline and functions are encapsulated in `src/preprocessing.py`.

The processed dataset is **fully validated, clean, and ready for Stage 5: Model Training**!
